# Exploring EW4 data - GPM IMERG
This notebook explore the GPM IMERG dataset, including loading in PyEarthTools.

This dataset is a subset of the [NASA Integrated Multi-satellite Retrieval (IMERG)](https://gpm.nasa.gov/data/imerg) data for Global Precipitation Measurement (GPM). This subset has the following extents:
- Temporal Extent: April to September 2025
- Spatial Extent
  - Latitude 0N to 20N
  - Longitude 27W to 20E


### Import libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pathlib
import datetime
import functools
import math

In [ ]:
import numpy

In [ ]:
import xarray

In [ ]:
import matplotlib
import cartopy.crs

In [ ]:
import site_archive_jasmin

In [ ]:
import pyearthtools

In [ ]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe

In [ ]:
from pyearthtools.data import Petdt, TimeDelta
from pyearthtools.data.exceptions import DataNotFoundError
from pyearthtools.data.indexes import ArchiveIndex, decorators
from pyearthtools.data.transforms import Transform, TransformCollection
from pyearthtools.data.archive import register_archive


In [ ]:
from site_archive_jasmin.utilities import (
    cached_exists,
    cached_iterdir,
)  # Could these be moved into a generic module?


In [ ]:
import torch

## Explore the data in the dataset
Lets start by looking at a sample data file, before demonstrating loading the data in pyearthtools. From this we can learn the following things about the dataset
* The pattern of filenames for accessing the data
* The spatial and tempoeral extents of the data
* The variables present in the data
* Any potential problems with its usage, which can be fixed in the PyEarthTools data accessor class
* Plot some data so we can check that PyEarthTools loads it correctly.

We'll be using the IMERG data in the EW4 Group Workspace. It is also available through other sources:
* [NASA](https://gpm.nasa.gov/data/imerg)
* [AWS Open Data](https://registry.opendata.aws/nasa-gpm3imergm/)
* [CEDA Archive - available on JASMIN](https://catalogue.ceda.ac.uk/uuid/47c32530265d4d6e8fdb6c08b2330371/)


In [ ]:
ew4_imerg_accessor = site_archive_jasmin.Ew4Imerg('2025-04-01 00:00', '2025-10-01 00:00')

In [ ]:
ghana_extents = {
    'latitude': (4.7,11.1),
    'longitude': (-3.5, 2.9),
}


In [ ]:
ghana_pet_box = (ghana_extents['latitude'][0],
                 ghana_extents['latitude'][1],
                 ghana_extents['longitude'][0],
                 ghana_extents['longitude'][1],
                )

### Adapting to produce a forecast
Here we are using a basic ML architecture, and we're not supply sufficient meteorological data to produce a good forecast, but we can demonstrate a simple change in pyearthtools that can switch from autoencoder to forecast model.

Change's to make are
* Change `TemporalWindow` arguments to point get a future state as the target
* Join existing pipeline components with new `TemporalRetrieval` to form new train and validate pipelines
* Rerun training

In [ ]:
train_forecast_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[-1,], posterior_indexes=[0,], timedelta=TimeDelta('30 minutes')),
    iterator=petpipe.iterators.DateRange('20250501T00', '20250701T00', interval='1 hour').randomise(), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)
val_forecast_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[-1,], posterior_indexes=[0,], timedelta=TimeDelta('30 minutes')),
    iterator=petpipe.iterators.DateRange('20250701T00', '20250801T00', interval='1 hour'), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

In [ ]:
ew4_imerg_train_forecast_pipe = ew4_imerg_prep | ew4_imerg_ml | train_range
ew4_imerg_val_forecast_pipe = ew4_imerg_prep | ew4_imerg_ml | val_range

In [ ]:
imerg_forecast = ImergAutoEncoder(1, False).to(device)

In [ ]:
num_samples_fc = len(ew4_imerg_train_forecast_pipe)
num_batches_fc = math.ceil(num_samples_fc / batch_size)
num_batches_fc

In [ ]:
for epoch_num in range(num_epochs):
    print(epoch_num)
    epoch_train_fc_loss = 0.0
    epoch_val_fc_loss = 0.0
    
    imerg_train_iter = iter(ew4_imerg_train_pipe)
    
    for batch_ix in range(num_batches):

        predictor_gpu_tensor, target_gpu_tensor = get_batch_tensors(batch_size, imerg_train_iter, device)
        
        if (batch_ix % 100) == 0:
            print(batch_ix)

        # do training for batch
        optimizer.zero_grad()
        predictions = imerg_forecast.forward(predictor_gpu_tensor)
        loss_batch = loss_function(predictions, target_gpu_tensor)
        loss_batch.backward()
        optimizer.step()
        epoch_train_loss += loss_batch.to('cpu').item()
    epoch_train_fc_loss /= num_batches

    # calculate loss on validation data    
    for val_predictor, val_target in ew4_imerg_val_pipe:
        val_pred_tensor = torch.tensor(val_predictor[0], dtype=torch.float32).to(device)
        predictions_val = imerg_forecast.forward( val_pred_tensor)
        val_target_tensor = torch.tensor(val_target[0], dtype=torch.float32).to(device)
        loss_batch_val = loss_function(predictions_val, val_target_tensor)
        epoch_val_fc_loss += loss_batch_val.to('cpu').item()
    
    epoch_val_fc_loss /= len(ew4_imerg_val_pipe)
    
    print('train loss:',epoch_train_fc_loss)
    print('validation loss:',epoch_val_fc_loss)

## Evaluate forecasts using Scores

In [ ]:
val_fc_rmse_list = []
for predictor, target in ew4_imerg_val_forecast_pipe:
    val_fc_arr = imerg_forecast.forward(torch.tensor(predictor[0], dtype=torch.float32).to(device)).to("cpu").detach().numpy()
    val_fc_rmse_list += [scores.continuous.rmse(predictor[0],
                                             val_fc_arr)]

In [ ]:
val_fc_rmse_list = numpy.mean(val_rmse_list)

In [ ]:
val_fc_rmse_list

In [ ]:
 predictor_fc, target_fc = ew4_imerg_val_forecast_pipe['2025-07-23 15:00']

In [ ]:
pred_fc_gpu = target_gpu = torch.tensor(
    predictor_fc[0],
    dtype=torch.float32,
).to(device)
pred_fc_gpu = torch.tensor(
    target_fc[0],
    dtype=torch.float32,
).to(device)

In [ ]:
model_forecast_array = imerg_forecast.forward(pred_fc_gpu).to("cpu").detach().numpy()

Our model prediction will be output as a numpy array (after we've extracted it from pytorch tensor). We want to then reinstate the metadata so we can interact with the model predictions, for example showing plots. We can use the pipeline reverse functionality to reverse the to numpy operation and get an xarray dataset out.

In [ ]:
input_fc_ds = ew4_imerg_ml.reversed(predictor_fc[0])
truth_fc_ds = ew4_imerg_ml.reversed(target_fc[0])
model_forecast = ew4_imerg_ml.reversed(model_forecast_array)

In [ ]:
(model_forecast['precipitation'] - truth_fc_ds['precipitation']).plot.hist()

In [ ]:
plot_kwargs = {'cmap':'viridis', 'vmin':0.0, 'vmax':5.0}

In [ ]:
(model_forecast['precipitation'] - truth_fc_ds['precipitation'])[0].plot.contourf(**plot_kwargs)

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(18,6))

ax1=fig1.add_subplot(1,3,1,projection=cartopy.crs.PlateCarree())
input_fc_ds['precipitation'][0].plot.contourf(ax=ax1, **plot_kwargs)
ax1.coastlines(color='w')

ax1=fig1.add_subplot(1,3,2,projection=cartopy.crs.PlateCarree())
model_forecast['precipitation'][0].plot.contourf(ax=ax1, **plot_kwargs)
ax1.coastlines(color='w')


ax1=fig1.add_subplot(1,3,3,projection=cartopy.crs.PlateCarree())
truth_fc_ds['precipitation'][0].plot.contourf(ax=ax1, **plot_kwargs)
ax1.coastlines(color='w')


## Further Links

* [PyEarthTools Docs](https://pyearthtools.readthedocs.io/en/latest/)
  * [Tutorial Gallery](https://pyearthtools.readthedocs.io/en/latest/notebooks/Gallery.html)
* [PyEarthTools Repo](https://github.com/ACCESS-Community-Hub/PyEarthTools)
* [PyEarthTools JASMIN Site Archive Repo](https://github.com/MetOffice/pyearthtools_jasmin/)
*  [IMERG Dataset Info](https://gpm.nasa.gov/data/imerg)